# Zerobus Ingest (gRPC + HTTP)

## Instructions:
- Clone/import the notebook into your Databricks workspace
- Select from the widgets:
  - **api** (`grpc` | `http/1.1` | `http/2`)
  - **sync_async** (`sync` | `async`)
  - **concurrency** (`1` | `2` | `4` | `8` | `16` | `32` | `64`)
    - only effective for `async` modes; ignored for `sync`
- Select serverless compute
- Click Run All

## Data path — no customer-managed queue

```
Producer                        Databricks                   Consumer    
+----------+          +---------------------------+          +----------+
| App w/   |  append  | Zerobus     Unity Catalog | full DML | AI       |
| Zerobus  |--------->| Endpoint -> Delta Table   |<---------| BI       |
| SDK/HTTP |          |                           |          | Agents   |
+----------+          +---------------------------+          +----------+
```

*No Kafka / Event Hubs / Kinesis queue you operate between the app and the managed Zerobus endpoint.*

## Setup

- Auto-configure Zerobus endpoint
- Auto-configure Service Principal (admin priv required to create SP if not present)
- Catalog, Schema, Table permissions

## Show performance characteristics
  - **~200 ms/row** for single-row inserts
  - **greater than ~200 ms+** for whole-batch insert
  - **~5 s** until all rows are visible

## Show system tables
  - system.lakeflow.zerobus_ingest (rows and bytes)
  - system.lakeflow.zerobus_stream (connections)

### More links
- [Zerobus overview](https://docs.databricks.com/aws/en/ingestion/zerobus-overview)
- [Python SDK repository](https://github.com/databricks/zerobus-sdk-py)
- [Get workspace URL and Zerobus endpoint](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#get-your-workspace-url-and-zerobus-ingest-endpoint)

# Step 1: Prepare the environment

In [0]:
%pip install --quiet databricks-zerobus-ingest-sdk aiohttp httpx[http2]

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.dropdown("api", "grpc", choices=[
    'grpc',
    'http/1.1',
    'http/2'
    ])

dbutils.widgets.dropdown("sync_async", "sync", choices=[
    'sync', 'async'])

dbutils.widgets.dropdown("concurrency", "1", choices=[
    '1', '2', '4', '8', '16', '32', '64'])


# Step 2: Configuration

## Step 2.a: Workspace and Zerobus endpoint (auto)

- `w.metastores.summary()` provides `.cloud` (→ cloud domain) and `.region`
- `w.get_workspace_id()` provides the workspace ID
- ZeroBus endpoint: `https://{workspace_id}.zerobus.{region}.{cloud_domain}`


In [0]:
import os
from databricks.sdk import WorkspaceClient

DATABRICKS_REGION_OVERRIDE = ""

_CLOUD_DOMAIN = {
    "AWS": "cloud.databricks.com",
    "AZURE": "azuredatabricks.net",
    "GCP": "gcp.databricks.com",
}

# Connect sets metadata-service auth; clear so WorkspaceClient uses ~/.databrickscfg locally.
if (os.environ.get("DATABRICKS_AUTH_TYPE") or "").strip().lower() == "metadata-service":
    os.environ.pop("DATABRICKS_AUTH_TYPE", None)
    os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
    print("Cleared metadata-service auth env vars for WorkspaceClient() (Connect-only; SDK will use your Databricks CLI / profile auth).")

_w = WorkspaceClient()
DATABRICKS_WORKSPACE_URL = _w.config.host.rstrip("/")
DATABRICKS_WORKSPACE_ID = str(_w.get_workspace_id())
DATABRICKS_WORKSPACE_O_QUERY = f"?o={DATABRICKS_WORKSPACE_ID}"
DATABRICKS_WORKSPACE_SP_UI_PREFIX = (
    f"{DATABRICKS_WORKSPACE_URL}/settings/workspace/identity-and-access/service-principals"
)

_summary = _w.metastores.summary()
DATABRICKS_REGION = (DATABRICKS_REGION_OVERRIDE or "").strip() or _summary.region
_domain = _CLOUD_DOMAIN.get((_summary.cloud or "").upper())
if not DATABRICKS_REGION or not _domain:
    raise RuntimeError(f"Unexpected metastore cloud={_summary.cloud!r} or empty region. Set DATABRICKS_REGION_OVERRIDE.")

ZEROBUS_INGEST_URL = f"https://{DATABRICKS_WORKSPACE_ID}.zerobus.{DATABRICKS_REGION}.{_domain}"
SERVER_ENDPOINT = ZEROBUS_INGEST_URL
print(f"{DATABRICKS_WORKSPACE_ID=}\n{DATABRICKS_WORKSPACE_URL=}\n{DATABRICKS_REGION=}\n{ZEROBUS_INGEST_URL=}\n{SERVER_ENDPOINT=}")

DATABRICKS_WORKSPACE_ID='1444828305810485'
DATABRICKS_WORKSPACE_URL='https://e2-demo-field-eng.cloud.databricks.com'
DATABRICKS_REGION='us-west-2'
ZEROBUS_INGEST_URL='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'
SERVER_ENDPOINT='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'


In [0]:
# Derive MODE from widgets. Requires Step 2.a (CATALOG/TABLE_NAME not yet set; just reads widget values).
_api = dbutils.widgets.get("api")           # "grpc" | "http/1.1" | "http/2"
_sync_async = dbutils.widgets.get("sync_async")   # "sync" | "async"

_MODE_MAP = {
    ("grpc",    "sync"):  "grpc_sync",
    ("grpc",    "async"): "grpc_async",
    ("http/1.1","sync"):  "http_sync",
    ("http/1.1","async"): "http_async",
    ("http/2",  "sync"):  "http2_sync",   # httpx Client(http2=True) — multiplexed single connection
    ("http/2",  "async"): "http2_async",  # httpx AsyncClient(http2=True)
}
MODE = _MODE_MAP.get((_api, _sync_async))
if MODE is None:
    raise ValueError(f"Unsupported combination: api={_api!r} sync_async={_sync_async!r}")
CONCURRENCY = int(dbutils.widgets.get("concurrency"))

print(f"{_api=}  {_sync_async=}  \u2192  {MODE=}  {CONCURRENCY=}")

_api='http/2'  _sync_async='async'  →  MODE='http2_async'  CONCURRENCY=1


## Step 2.b: Service principal — create or retrieve

- `dbutils.secrets.get` reads saved SP from Databricks secrets
- else `_w.service_principals.create` creates SP and saves to Databricks secrets


In [0]:
# Step 2.b: Secret scope/key + service principal + initial JSON (no OAuth client secret mint here).

# Workspace-specific prefix; optional tail: --<scope>--<secretKey>--<oauthJsonField>
SP_NAME = "lfcdemo_zerobus"

_DEFAULT_SP_SECRET_SCOPE = "lfczerobusdemo"
_DEFAULT_SP_SECRET_KEY = "lfczerobusdemo"
_DEFAULT_SP_OAUTH_JSON_FIELD = "ZEROBUS_OAUTH_SECRET"

import json
import re

from databricks.sdk.errors import NotFound, ResourceAlreadyExists, ResourceDoesNotExist


def _looks_like_oauth_json_key(s: str) -> bool:
    """Legacy 3-part refs use an UPPER_SNAKE json field as the third segment."""
    return bool(re.fullmatch(r"[A-Z][A-Z0-9_]*", s))


def _parse_secret_ref(ref: str):
    """Parse scope--secretKey--jsonField or prefix--scope--secretKey--jsonField.

    With a leading workspace prefix, trailing segments may be omitted: missing scope,
    secret key, and oauth json field default to _DEFAULT_SP_SECRET_SCOPE,
    _DEFAULT_SP_SECRET_KEY, and _DEFAULT_SP_OAUTH_JSON_FIELD respectively.

    A 3-segment value is either that legacy form (third segment looks like a json field
    name) or prefix--scope--secretKey with the default oauth json field.
    """
    parts = [p for p in str(ref).strip().split("--") if p != ""]
    if not parts:
        raise ValueError("SP_NAME is empty or only contains '--' separators.")
    if len(parts) > 4:
        raise ValueError(
            "Secret locator: at most 4 '--' segments "
            "(prefix--scope--secretKey--jsonField, or scope--secretKey--jsonField); "
            f"got {ref!r}"
        )
    if len(parts) == 1:
        return (
            _DEFAULT_SP_SECRET_SCOPE,
            _DEFAULT_SP_SECRET_KEY,
            _DEFAULT_SP_OAUTH_JSON_FIELD,
        )
    if len(parts) == 2:
        return parts[1], _DEFAULT_SP_SECRET_KEY, _DEFAULT_SP_OAUTH_JSON_FIELD
    if len(parts) == 3:
        if _looks_like_oauth_json_key(parts[2]):
            return parts[0], parts[1], parts[2]
        return parts[1], parts[2], _DEFAULT_SP_OAUTH_JSON_FIELD
    return parts[1], parts[2], parts[3]


if not str(SP_NAME).strip():
    raise ValueError("SP_NAME is empty.")

_SECRET_SCOPE, _SECRET_KEY, _oauth_field = _parse_secret_ref(SP_NAME)
_config_keys = [
    "ZEROBUS_SERVICE_PRINCIPAL_NAME",
    "ZEROBUS_SERVICE_PRINCIPAL_ID",
    "ZEROBUS_APP_ID",
    _oauth_field,
]


def _put_secret_json(blob: dict) -> None:
    """Write the full JSON value for this scope/key (shared with Step 2.c)."""
    _w.secrets.put_secret(
        scope=_SECRET_SCOPE,
        key=_SECRET_KEY,
        string_value=json.dumps(blob, indent=2),
    )


def _ensure_secret_scope_and_key() -> None:
    try:
        _w.secrets.create_scope(_SECRET_SCOPE)
        print(f"Created secret scope {_SECRET_SCOPE!r}")
    except ResourceAlreadyExists:
        pass
    try:
        _w.secrets.get_secret(_SECRET_SCOPE, _SECRET_KEY)
    except ResourceDoesNotExist:
        _w.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value="{}")
        print(f"Created empty secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r}")


def _load_saved() -> dict:
    try:
        raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY)
        print(f"Loaded config from secret scope={_SECRET_SCOPE!r} key={_SECRET_KEY!r}")
        return json.loads(raw)
    except Exception:
        pass
    _ensure_secret_scope_and_key()
    return {}


def _save_config(updates: dict) -> None:
    try:
        current = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
    except ResourceDoesNotExist:
        current = {}
    current.update(updates)
    _put_secret_json(current)
    print(f"Saved {list(updates.keys())} to secret {_SECRET_SCOPE}/{_SECRET_KEY}")


_saved = _load_saved()
_config = {k: _saved.get(k, "") for k in _config_keys}
_config_original = {k: _saved.get(k, "") for k in _config_keys}

_sp_state = {"replaced_stale_sp": False}


def _ensure_sp() -> None:
    sp_id = _config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")

    def _create_sp():
        _name = str(SP_NAME).strip()
        _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = _name
        _sp = _w.service_principals.create(display_name=_name)
        _config["ZEROBUS_SERVICE_PRINCIPAL_ID"] = str(_sp.id)
        _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
        print(f"Created SP {_name!r} id={_sp.id} APP_ID={_config['ZEROBUS_APP_ID']}")

    if not str(sp_id).strip():
        print("ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal")
        _create_sp()
        return

    try:
        _sp = _w.service_principals.get(sp_id)
        print(f"SP exists: {_sp.display_name!r} sp_id={sp_id}")
        api_app = str(_sp.application_id)
        stored_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()
        if stored_app and stored_app != api_app:
            raise RuntimeError(
                f"Secret ZEROBUS_APP_ID={stored_app!r} does not match workspace SP application_id={api_app!r} "
                f"for sp_id={sp_id!r}. Fix the secret JSON or the service principal."
            )
        if not stored_app:
            _config["ZEROBUS_APP_ID"] = api_app
            print(f"Backfilled ZEROBUS_APP_ID={_config['ZEROBUS_APP_ID']}")
    except NotFound:
        print(f"SP id={sp_id!r} not found — creating a replacement")
        _sp_state["replaced_stale_sp"] = True
        _create_sp()


_ensure_sp()

_o_sp = str(_config_original.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_o_app = str(_config_original.get("ZEROBUS_APP_ID", "")).strip()
_n_sp = str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_n_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()

# Once both ids were stored, they must not drift unless we replaced a deleted SP (stale id in the secret).
if _o_sp and _o_app and not _sp_state["replaced_stale_sp"]:
    if _n_sp != _o_sp or _n_app != _o_app:
        raise RuntimeError(
            "Refusing to persist: ZEROBUS_SERVICE_PRINCIPAL_ID or ZEROBUS_APP_ID would change. "
            f"stored sp_id={_o_sp!r} app_id={_o_app!r}; after Step 2.b got sp_id={_n_sp!r} app_id={_n_app!r}. "
            "If you rotated the service principal, update or clear the secret JSON manually."
        )

# Persist when we first record SP metadata, backfill a missing APP_ID, replace a deleted SP, or align name/oauth keys.
# Steady re-runs with the same JSON produce no updates (not an error—just no write).
_updates = {k: _config[k] for k in _config_keys if _config.get(k) != _config_original.get(k)}
if _updates:
    _save_config(_updates)

if not str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip():
    raise RuntimeError(
        "Step 2.b: ZEROBUS_SERVICE_PRINCIPAL_ID is still empty after bootstrap; cannot continue."
    )
if not str(_config.get("ZEROBUS_APP_ID", "")).strip():
    raise RuntimeError("Step 2.b: ZEROBUS_APP_ID is empty after bootstrap; cannot continue.")

Loaded config from secret scope='lfczerobusdemo' key='lfczerobusdemo'
SP exists: 'lfcdemo_zerobus_sp' sp_id=75332893425169


In [0]:
# After Step 2.b: admin UI for this service principal (DATABRICKS_WORKSPACE_* from Step 2.a).
_sp = str(_config["ZEROBUS_SERVICE_PRINCIPAL_ID"]).strip()
print(f"{DATABRICKS_WORKSPACE_SP_UI_PREFIX}/{_sp}{DATABRICKS_WORKSPACE_O_QUERY}")

https://e2-demo-field-eng.cloud.databricks.com/settings/workspace/identity-and-access/service-principals/75332893425169?o=1444828305810485


## Step 2.c: OAuth client secret — validate or mint

- `dbutils.secrets.get` reads stored SP config from the same secret as Step 2.b
- `POST {workspace}/oidc/v1/token` (`client_credentials`) validates the stored secret
- If invalid or missing: `service_principal_secrets_proxy.create` mints a new secret and saves back to Databricks secrets


In [0]:
# Step 2.c: Read client secret from the Step 2.b secret; OIDC-validate; mint + persist if invalid.
# Requires Step 2.b (defines _SECRET_SCOPE, _SECRET_KEY, _oauth_field, _put_secret_json) and Step 2.a (_w).

import json
import urllib.error
import urllib.request
from urllib.parse import urlencode


def _credentials_valid(client_id: str, client_secret: str, workspace_url: str) -> bool:
    """True if client_id + client_secret work against the workspace OIDC token endpoint."""
    if not client_id or not client_secret:
        return False
    try:
        body = urlencode(
            {
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            }
        ).encode()
        req = urllib.request.Request(
            f"{workspace_url.rstrip('/')}/oidc/v1/token",
            data=body,
            method="POST",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            return 200 <= getattr(resp, "status", 200) < 300
    except urllib.error.HTTPError as ex:
        try:
            msg = ex.read().decode(errors="replace")
        except Exception:
            msg = str(ex)
        print(f"OIDC check: HTTP {ex.code} {msg[:500]}")
        return False
    except Exception as ex:
        print(f"OIDC check: {ex}")
        return False


for _need in ("_SECRET_SCOPE", "_SECRET_KEY", "_oauth_field", "_put_secret_json"):
    if _need not in globals():
        raise RuntimeError(
            f"Step 2.c requires Step 2.b first (missing {_need}). Re-run Step 2.b after restarting the kernel if needed."
        )

_blob = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
if "ZEROBUS_APP_ID" not in _blob:
    raise KeyError(
        "Secret JSON must include ZEROBUS_APP_ID (OAuth application id). Run Step 2.b to create the service principal."
    )
if _oauth_field not in _blob:
    _blob[_oauth_field] = ""

client_id = str(_blob["ZEROBUS_APP_ID"]).strip()
client_secret = str(_blob.get(_oauth_field, "") or "").strip()
_ws_url = _w.config.host.rstrip("/")

if _credentials_valid(client_id, client_secret, _ws_url):
    print('CLIENT_SECRET="***" OAuth client secret is valid (OIDC client_credentials).')
else:
    print("OAuth client secret missing or invalid — minting a new client secret\u2026")
    sp_id = str(_blob.get("ZEROBUS_SERVICE_PRINCIPAL_ID") or "").strip()
    if not sp_id:
        raise RuntimeError(
            "Cannot mint OAuth client secret: ZEROBUS_SERVICE_PRINCIPAL_ID missing in secret JSON. Run Step 2.b."
        )
    _secret_obj = None
    try:
        _secret_obj = _w.service_principal_secrets_proxy.create(service_principal_id=sp_id)
    except AttributeError:
        try:
            from databricks.sdk import ServicePrincipalSecretsAPI

            _secret_obj = ServicePrincipalSecretsAPI(_w.api_client).create(service_principal_id=sp_id)
        except Exception:
            _resp = _w.api_client.do(
                "POST",
                f"/api/2.0/accounts/servicePrincipals/{sp_id}/credentials/secrets",
            )
            _blob[_oauth_field] = _resp["secret"]

    if _secret_obj is not None:
        _blob[_oauth_field] = _secret_obj.secret

    client_secret = str(_blob.get(_oauth_field, "") or "").strip()
    if not _credentials_valid(client_id, client_secret, _ws_url):
        raise RuntimeError(
            "New client secret failed OIDC check; verify workspace URL and SP permissions."
        )
    _put_secret_json(_blob)
    print(
        f"Saved new OAuth client secret to secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r} (field {_oauth_field!r})."
    )

CLIENT_ID = client_id
CLIENT_SECRET = client_secret
if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Step 2.c: CLIENT_ID or CLIENT_SECRET is empty after load/mint.")

print(f"{CLIENT_ID=}")

CLIENT_SECRET="***" OAuth client secret is valid (OIDC client_credentials).
CLIENT_ID='e6d2f259-8c72-4ac4-826b-f773af1528fc'


In [0]:
# After Step 2.c: SP secrets (OAuth) in workspace UI.
print(
    f"{DATABRICKS_WORKSPACE_SP_UI_PREFIX}/{str(_config['ZEROBUS_SERVICE_PRINCIPAL_ID']).strip()}/secrets{DATABRICKS_WORKSPACE_O_QUERY}"
)

https://e2-demo-field-eng.cloud.databricks.com/settings/workspace/identity-and-access/service-principals/75332893425169/secrets?o=1444828305810485


## Step 2.d: Catalog, schema, and table

- `TABLE = f"airquality_{MODE}"` — one table per ingest mode
- `spark.sql("SELECT current_catalog()")` resolves `CATALOG`; falls back to `SHOW CATALOGS` if result is `hive_metastore` — ZeroBus requires a UC managed Delta table (not `hive_metastore`, returns error 4024)
- `_w.current_user.me().user_name` derives default `SCHEMA` from the Databricks username
- Builds `TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"` (UC 3-part name)


In [0]:
# Table (Step 2.d). Table name is derived from MODE so each mode has its own table.
# Zerobus requires a UC *managed* Delta table — not hive_metastore / default storage (gRPC 4024).
CATALOG = ""  # blank → Spark default, avoiding hive_metastore when possible; or set explicitly, e.g. "main"
SCHEMA = ""  # blank → UC schema from Databricks username (e.g. robert.lee@… → robert_lee)
TABLE = f"airquality_{MODE}"  # airquality_grpc_sync | airquality_grpc_async | airquality_http_sync | airquality_http_async | airquality_http2_sync | airquality_http2_async

_ZB_EXCLUDED_CATALOGS = frozenset({"", "hive_metastore", "spark_catalog"})


def _catalog_name_from_show_row(row) -> str:
    if hasattr(row, "catalogName"):
        return str(row.catalogName)
    d = row.asDict(recursive=True)
    for k in ("catalogName", "catalog", "namespace"):
        if k in d and d[k] is not None:
            return str(d[k])
    return str(row[0])


def _default_catalog_for_zerobus() -> str:
    cur = spark.sql("SELECT current_catalog()").collect()[0][0]
    if str(cur).strip().lower() not in _ZB_EXCLUDED_CATALOGS:
        return str(cur)
    names = []
    for row in spark.sql("SHOW CATALOGS").collect():
        n = _catalog_name_from_show_row(row).strip()
        if n.lower() not in _ZB_EXCLUDED_CATALOGS:
            names.append(n)
    if not names:
        raise RuntimeError(
            "Zerobus Ingest needs a Unity Catalog catalog (managed storage). current_catalog() is "
            f"{cur!r} and no other catalogs are listed. Set CATALOG explicitly in this cell."
        )
    _main = next((x for x in names if x.lower() == "main"), None)
    if _main is not None:
        print(f"current_catalog was {cur!r}; using {_main!r} for UC-managed storage (Zerobus).")
        return _main
    names.sort(key=str.lower)
    print(f"current_catalog was {cur!r}; using {names[0]!r} for UC-managed storage (Zerobus).")
    return names[0]


if not str(CATALOG).strip():
    CATALOG = _default_catalog_for_zerobus()
    print(f"CATALOG default: {CATALOG!r}")

if not str(SCHEMA).strip():
    SCHEMA = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0].lower())
    print(f"SCHEMA default (from current user): {SCHEMA!r}")

TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

CATALOG default: 'main'
SCHEMA default (from current user): 'robert_lee'


# Step 3: Create table and grant SP access

- `CREATE SCHEMA IF NOT EXISTS` + `CREATE TABLE IF NOT EXISTS` — idempotent; schema created first
- Grants `CLIENT_ID` (SP `application_id`) minimum permissions: `USE CATALOG`, `USE SCHEMA`, `MODIFY + SELECT ON TABLE`


In [0]:
# DDL then SP grants (grants require schema + table to exist).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
  device_name STRING,
  temp        INT,
  humidity    BIGINT
) USING DELTA
""")
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `" + CLIENT_ID + "`;" ).collect()
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `" + CLIENT_ID + "`;" ).collect()
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `" + CLIENT_ID + "`;" ).collect()

[]

In [0]:
# Open this table in workspace Data Explorer (Unity Catalog).
from urllib.parse import quote

print(
    f"{DATABRICKS_WORKSPACE_URL}/explore/data/"
    f"{quote(str(CATALOG), safe='')}/{quote(str(SCHEMA), safe='')}/{quote(str(TABLE), safe='')}"
    f"{DATABRICKS_WORKSPACE_O_QUERY}"
)

https://e2-demo-field-eng.cloud.databricks.com/explore/data/main/robert_lee/airquality_http2_async?o=1444828305810485


# Step 4: Ingest helpers — `setup_zerobus` and `call_zerobus_insert`

`setup_zerobus(mode)` opens the connection for the given mode:
- gRPC: `sdk.create_stream(CLIENT_ID, CLIENT_SECRET, ...)` opens the stream
- HTTP: `POST /oidc/v1/token` with `authorization_details` scoping UC privileges (CATALOG / SCHEMA / TABLE) to the SP; one warm-up GET establishes TCP+TLS (HTTP/2: + ALPN)

`call_zerobus_insert(client, rows)` sends rows and returns `(send_s, wait_s)`:
- gRPC: `ingest_record_offset` + `wait_for_offset` — send and ack wait timed separately
- HTTP: `session.post(rest_url, json.dumps(rows))` — synchronous round-trip, `wait_s` always `0.0`


In [0]:
# Helper functions — run once after Step 2.d.
# Requires: SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL, TABLE_NAME, CATALOG, SCHEMA,
#            DATABRICKS_WORKSPACE_ID, ZEROBUS_INGEST_URL, CLIENT_ID, CLIENT_SECRET.

import json as _json
import time as _time


async def setup_zerobus(mode: str) -> dict:
    """Open a Zerobus stream or HTTP session for the given mode.

    Returns a client dict with keys:
      mode, stream, session, close, _connect_s, _ack_events (gRPC only),
      _access_token / _rest_insert_url (HTTP only).
    """
    if mode == "grpc_sync":
        from zerobus.sdk.sync import ZerobusSdk
        from zerobus.sdk.shared import AckCallback, RecordType, StreamConfigurationOptions, TableProperties

        _ack_events = []

        class _DemoAckCallback(AckCallback):
            def on_ack(self, offset: int) -> None:
                print(f"[ack callback] offset {offset} acknowledged")
                _ack_events.append(("ack", offset))

            def on_error(self, offset: int, error_message: str) -> None:
                print(f"[ack callback] error at offset {offset}: {error_message}")
                _ack_events.append(("error", offset, error_message))

        sdk = ZerobusSdk(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL)
        table_properties = TableProperties(TABLE_NAME)
        options = StreamConfigurationOptions(
            record_type=RecordType.JSON,
            ack_callback=_DemoAckCallback(),
        )
        t0 = _time.perf_counter()
        stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)
        connect_s = _time.perf_counter() - t0

        async def _close():
            stream.close()

        return {
            "mode": mode, "stream": stream, "session": None, "close": _close,
            "_connect_s": connect_s, "_oauth_s": None, "_ack_events": _ack_events,
        }

    elif mode == "grpc_async":
        try:
            from zerobus.sdk.aio import ZerobusSdkAsync
        except ImportError:
            from zerobus.sdk.aio import ZerobusSdk as ZerobusSdkAsync
        from zerobus.sdk.shared import AckCallback, RecordType, StreamConfigurationOptions, TableProperties

        _ack_events = []

        class _DemoAckCallback(AckCallback):
            def on_ack(self, offset: int) -> None:
                print(f"[ack callback] offset {offset} acknowledged")
                _ack_events.append(("ack", offset))

            def on_error(self, offset: int, error_message: str) -> None:
                print(f"[ack callback] error at offset {offset}: {error_message}")
                _ack_events.append(("error", offset, error_message))

        sdk = ZerobusSdkAsync(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL)
        table_properties = TableProperties(TABLE_NAME)
        options = StreamConfigurationOptions(
            record_type=RecordType.JSON,
            ack_callback=_DemoAckCallback(),
        )
        t0 = _time.perf_counter()
        stream = await sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)
        connect_s = _time.perf_counter() - t0

        async def _close():
            await stream.close()

        return {
            "mode": mode, "stream": stream, "session": None, "close": _close,
            "_connect_s": connect_s, "_oauth_s": None, "_ack_events": _ack_events,
        }

    elif mode in ("http_sync", "http_async", "http2_sync", "http2_async"):
        import requests
        from requests.auth import HTTPBasicAuth

        _authorization_details = _json.dumps([
            {
                "type": "unity_catalog_privileges",
                "privileges": ["USE CATALOG"],
                "object_type": "CATALOG",
                "object_full_path": CATALOG,
            },
            {
                "type": "unity_catalog_privileges",
                "privileges": ["USE SCHEMA"],
                "object_type": "SCHEMA",
                "object_full_path": f"{CATALOG}.{SCHEMA}",
            },
            {
                "type": "unity_catalog_privileges",
                "privileges": ["SELECT", "MODIFY"],
                "object_type": "TABLE",
                "object_full_path": TABLE_NAME,
            },
        ])
        t0_oauth = _time.perf_counter()
        _token_resp = requests.post(
            f"{DATABRICKS_WORKSPACE_URL}/oidc/v1/token",
            auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
            data={
                "grant_type": "client_credentials",
                "scope": "all-apis",
                "resource": f"api://databricks/workspaces/{DATABRICKS_WORKSPACE_ID}/zerobusDirectWriteApi",
                "authorization_details": _authorization_details,
            },
            timeout=60,
        )
        _token_resp.raise_for_status()
        access_token = _token_resp.json()["access_token"]
        oauth_s = _time.perf_counter() - t0_oauth
        print(f"Fetched OAuth access token in {oauth_s * 1000:.1f} ms")
        rest_url = f"{ZEROBUS_INGEST_URL}/zerobus/v1/tables/{TABLE_NAME}/insert"

        if mode == "http_sync":
            # HTTP/1.1 — requests.Session with keep-alive; connections pooled per host.
            session = requests.Session()
            # Force TCP+TLS handshake now so 4a/4b use a warm connection (analogous to gRPC create_stream).
            t0 = _time.perf_counter()
            try:
                session.get(ZEROBUS_INGEST_URL, timeout=30)
            except Exception:
                pass  # any response (even 4xx) establishes the connection
            connect_s = _time.perf_counter() - t0
            print(f"[http_sync] TCP+TLS connected in {connect_s * 1000:.1f} ms")

            async def _close():
                session.close()

        elif mode == "http_async":
            # HTTP/1.1 async — aiohttp ClientSession; connection pool reused across awaits.
            import aiohttp
            session = aiohttp.ClientSession()
            # Force TCP+TLS handshake now so 4a/4b use a warm connection.
            t0 = _time.perf_counter()
            try:
                async with session.get(ZEROBUS_INGEST_URL, timeout=aiohttp.ClientTimeout(total=30)) as _r:
                    pass
            except Exception:
                pass
            connect_s = _time.perf_counter() - t0
            print(f"[http_async] TCP+TLS connected in {connect_s * 1000:.1f} ms")

            async def _close():
                await session.close()

        elif mode == "http2_sync":
            # HTTP/2 sync — httpx Client with a single multiplexed connection; no per-request TCP overhead.
            import httpx
            session = httpx.Client(http2=True)
            # Force TCP+TLS+HTTP/2 negotiation now.
            t0 = _time.perf_counter()
            try:
                session.get(ZEROBUS_INGEST_URL, timeout=30)
            except Exception:
                pass
            connect_s = _time.perf_counter() - t0
            print(f"[http2_sync] TCP+TLS+h2 connected in {connect_s * 1000:.1f} ms")

            async def _close():
                session.close()

        else:  # http2_async
            # HTTP/2 async — httpx AsyncClient; multiple in-flight streams over one connection.
            import httpx
            session = httpx.AsyncClient(http2=True)
            # Force TCP+TLS+HTTP/2 negotiation now.
            t0 = _time.perf_counter()
            try:
                await session.get(ZEROBUS_INGEST_URL, timeout=30)
            except Exception:
                pass
            connect_s = _time.perf_counter() - t0
            print(f"[http2_async] TCP+TLS+h2 connected in {connect_s * 1000:.1f} ms")

            async def _close():
                await session.aclose()

        return {
            "mode": mode, "stream": None, "session": session, "close": _close,
            "_connect_s": connect_s, "_oauth_s": oauth_s,
            "_access_token": access_token, "_rest_insert_url": rest_url,
        }

    else:
        raise ValueError(
            f"Unknown mode: {mode!r}. Use 'grpc_sync', 'grpc_async', 'http_sync', 'http_async', 'http2_sync', or 'http2_async'."
        )


async def call_zerobus_insert(client: dict, rows: list) -> tuple:
    """Insert rows using the client returned by setup_zerobus.

    Returns (send_seconds, wait_seconds). For HTTP modes wait_seconds is always 0.0
    (the POST is a synchronous round-trip). For gRPC modes send_seconds covers the
    ingest call and wait_seconds covers wait_for_offset.
    """
    mode = client["mode"]

    if mode == "grpc_sync":
        stream = client["stream"]
        t_send = _time.perf_counter()
        offset = stream.ingest_record_offset(rows[0]) if len(rows) == 1 else stream.ingest_records_offset(rows)
        send_s = _time.perf_counter() - t_send
        t_wait = _time.perf_counter()
        stream.wait_for_offset(offset)
        return send_s, _time.perf_counter() - t_wait

    elif mode == "grpc_async":
        stream = client["stream"]
        t_send = _time.perf_counter()
        if len(rows) == 1:
            offset = await stream.ingest_record_offset(rows[0])
        else:
            offset = await stream.ingest_records_offset(rows)
        send_s = _time.perf_counter() - t_send
        t_wait = _time.perf_counter()
        await stream.wait_for_offset(offset)
        return send_s, _time.perf_counter() - t_wait

    elif mode == "http_sync":
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {client['_access_token']}",
        }
        t0 = _time.perf_counter()
        resp = client["session"].post(
            client["_rest_insert_url"],
            headers=headers,
            data=_json.dumps(rows),
            timeout=120,
        )
        resp.raise_for_status()
        return _time.perf_counter() - t0, 0.0

    elif mode == "http_async":
        import aiohttp
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {client['_access_token']}",
        }
        t0 = _time.perf_counter()
        async with client["session"].post(
            client["_rest_insert_url"],
            data=_json.dumps(rows),
            headers=headers,
            timeout=aiohttp.ClientTimeout(total=120),
        ) as resp:
            resp.raise_for_status()
        return _time.perf_counter() - t0, 0.0

    elif mode == "http2_sync":
        # httpx sync — HTTP/2 multiplexed; content= for raw bytes (vs data= in requests).
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {client['_access_token']}",
        }
        t0 = _time.perf_counter()
        resp = client["session"].post(
            client["_rest_insert_url"],
            content=_json.dumps(rows),
            headers=headers,
            timeout=120,
        )
        resp.raise_for_status()
        return _time.perf_counter() - t0, 0.0

    elif mode == "http2_async":
        # httpx async — HTTP/2 multiplexed; content= for raw bytes.
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {client['_access_token']}",
        }
        t0 = _time.perf_counter()
        resp = await client["session"].post(
            client["_rest_insert_url"],
            content=_json.dumps(rows),
            headers=headers,
            timeout=120,
        )
        resp.raise_for_status()
        return _time.perf_counter() - t0, 0.0

    else:
        raise ValueError(f"Unknown mode in client: {mode!r}")

## Step 4.a setup: baseline + open connection

- `_n` — row count for the run (lower for quick test)
- `spark.sql("SELECT COUNT(*)")` — row baseline before ingest; compared after Step 4.b to confirm rows landed
- `await setup_zerobus(MODE)` — opens connection; `_connect_s` = stream/TCP+TLS time, `_oauth_s` = OAuth fetch (HTTP only, `None` for gRPC)
- GET to `ZEROBUS_INGEST_URL` on warm connection — measures base HTTP round-trip latency
- Concurrency `> 1` is silently ignored for sync modes — blocking Rust/socket calls hold the event loop and serialize


In [0]:
import json
import logging
import time

logging.basicConfig(level=logging.INFO)

_n = 1000  # docs use 1000; lower for a quick test

# Baseline (small demo table: COUNT + MIN/MAX are cheap; no extra index needed)
_row_before = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
_bounds_before = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"Before ingest: count={_row_before} min(device_name)={_bounds_before['lo']!r} max(device_name)={_bounds_before['hi']!r}"
)

_zb_client = await setup_zerobus(MODE)
_connect_s = _zb_client["_connect_s"]  # None for HTTP modes before warmup was added
_oauth_s   = _zb_client["_oauth_s"]    # HTTP only; None for gRPC (OAuth inside Rust create_stream)

# Ping ZeroBus endpoint on the warm connection to measure base round-trip latency.
# Uses a plain GET to ZEROBUS_INGEST_URL (base URL, not the insert path) across all modes.
import requests as _req
_t_ping0 = time.perf_counter()
try:
    _req.get(ZEROBUS_INGEST_URL, timeout=10)
except Exception:
    pass  # any response (even 4xx) counts; we just want the round-trip time
_ping_s = time.perf_counter() - _t_ping0
print(f"Ping {ZEROBUS_INGEST_URL}: {_ping_s * 1000:.1f} ms")

_singles = min(10, _n)
_batch_runs = 10  # consecutive batch sends for 4b latency distribution

_concurrency = CONCURRENCY
if _concurrency > 1 and not MODE.endswith("_async"):
    print(f"Note: concurrency={_concurrency} has no effect for sync modes "
          f"(blocking Rust/socket calls hold the event loop — runs serialize).")
else:
    print(f"Concurrency: {_concurrency}")


def _json_payload_bytes(obj: dict) -> int:
    # UTF-8 length of JSON (compact); aligns with byte-oriented ingest / billing discussion
    return len(json.dumps(obj, separators=(",", ":")).encode("utf-8"))

Before ingest: count=921630 min(device_name)='sensor-0' max(device_name)='sensor-999'
Fetched OAuth access token in 277.6 ms
[http2_async] TCP+TLS+h2 connected in 28.8 ms
Ping https://1444828305810485.zerobus.us-west-2.cloud.databricks.com: 74.8 ms
Concurrency: 1


## Step 4.a: Single-row ingest (`_singles` rows, one row per call)

- `call_zerobus_insert` handles all protocol branching (gRPC sync/async, HTTP/1.1, HTTP/2) — same call site for all 6 modes
  - `asyncio.Semaphore(_concurrency)` caps concurrent in-flight inserts
  - `asyncio.gather(...)` dispatches all inserts concurrently, bounded by the semaphore
  - Records built upfront so byte counting is outside the timed section
  - `send_s + wait_s` = total per-call latency stored in `_ack_seconds`


In [0]:
import asyncio as _asyncio

_row_send_seconds = []
_row_wait_seconds = []
_row_ack_seconds = []
_singles_wall_s = 0.0
_bytes_4a = 0

# Build all records upfront so bytes can be counted before the timed section.
_records_4a = [
    {"device_name": f"sensor-{i}", "temp": 20 + i % 15, "humidity": 50 + i % 40}
    for i in range(_singles)
]
_bytes_4a = sum(_json_payload_bytes(r) for r in _records_4a)

_sem = _asyncio.Semaphore(_concurrency)

async def _insert_one(rec):
    async with _sem:
        return await call_zerobus_insert(_zb_client, [rec])

t_4a0 = time.perf_counter()
_results = await _asyncio.gather(*[_insert_one(r) for r in _records_4a])
_singles_wall_s = time.perf_counter() - t_4a0
for send_s, wait_s in _results:
    _row_send_seconds.append(send_s)
    _row_wait_seconds.append(wait_s)
    _row_ack_seconds.append(send_s + wait_s)

## Step 4.b: Batch ingest (rows `_singles` → `_n` as one call, repeated `_batch_runs` times)

- `call_zerobus_insert(_zb_client, batch_records)` — sends remaining rows as a single batch call (vs one row per call in 4a)
- `asyncio.gather(...)` repeats the same batch `_batch_runs` times concurrently to sample latency distribution
- `finally: _zb_client["close"]()` — connection closed here regardless; `_disconnect_s` is ~0 ms for HTTP (pool teardown), meaningful for gRPC


In [0]:
_batch_lo = _singles
_batch_run_seconds = []    # total wall time per batch run (send + wait)
_batch_send_seconds = []   # send portion per run
_batch_wait_seconds = []   # wait/ack portion per run (0.0 for HTTP modes)
_bytes_4b = 0
try:
    if _batch_lo < _n:
        batch_records = [
            {
                "device_name": f"sensor-{i}",
                "temp": 20 + i % 15,
                "humidity": 50 + i % 40,
            }
            for i in range(_batch_lo, _n)
        ]
        _bytes_4b = sum(_json_payload_bytes(r) for r in batch_records)
        async def _insert_batch():
            async with _sem:
                return await call_zerobus_insert(_zb_client, batch_records)

        _results = await _asyncio.gather(*[_insert_batch() for _ in range(_batch_runs)])
        for send_s, wait_s in _results:
            _batch_send_seconds.append(send_s)
            _batch_wait_seconds.append(wait_s)
            _batch_run_seconds.append(send_s + wait_s)
finally:
    _t_close_start = time.perf_counter()
    await _zb_client["close"]()
    _t_after_close = time.perf_counter()
    _disconnect_s = _t_after_close - _t_close_start  # all modes; ~0 ms for HTTP (pool teardown)

_batch_n = max(0, _n - _singles)
_total_rows_inserted = _singles + _batch_n * len(_batch_run_seconds)
_ingest_4a4b_s = _singles_wall_s + sum(_batch_run_seconds)
_bytes_payload_total = _bytes_4a + _bytes_4b

## Step 4.c: Lakehouse visibility

Poll `COUNT(*)` until ingested rows are visible in UC.

In [0]:
# Lakehouse visibility: table can lag a few seconds after SDK acks complete

_target_count = _row_before + _total_rows_inserted
_poll_deadline = _t_after_close + 120.0
_row_visible = _row_before
while time.perf_counter() < _poll_deadline:
    _row_visible = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
    if _row_visible >= _target_count:
        break
    time.sleep(0.15)

_t_visible = time.perf_counter()
_visibility_s = _t_visible - _t_after_close        # end of ingest → COUNT(*) target
_visibility_from_first_send_s = _t_visible - t_4a0  # first row send → COUNT(*) target

## Step 4.d: Key metrics

In [0]:
from statistics import mean, median

print(
    f"Before ingest: count={_row_before} min(device_name)={_bounds_before['lo']!r} max(device_name)={_bounds_before['hi']!r}"
)
print(f"Ingested {_n} rows into {TABLE_NAME}  [mode={MODE}]")
print(f"After ingest:  expected_count={_target_count} visible_count={_row_visible}")
print(
    f"  visibility: {_visibility_from_first_send_s * 1000:.1f} ms  (from first row send \u2192 COUNT(*) reached target)"
)
print(
    f"  visibility: {_visibility_s * 1000:.1f} ms  (from end of ingest \u2192 COUNT(*) reached target)"
)
if _row_visible < _target_count:
    print("  Warning: COUNT still short after poll window; re-run or raise poll budget.")

_bounds_after = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"After visibility: min(device_name)={_bounds_after['lo']!r} max(device_name)={_bounds_after['hi']!r}"
)

if _oauth_s is not None:
    print(f"oauth:        {_oauth_s * 1000:.1f} ms")
if _connect_s is not None:
    print(f"connect:      {_connect_s * 1000:.1f} ms")
if _disconnect_s is not None:
    print(f"disconnect:   {_disconnect_s * 1000:.1f} ms")
print(f"  Ingest+wait wall (4a+4b only; excludes stream.close): {_ingest_4a4b_s * 1000:.1f} ms")
if _singles:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    print(
        f"  4a ({_singles} singles): wall {_singles_wall_s * 1000:.1f} ms  (~{_ms_row_4a:.1f} ms/row amortized)"
    )
    print(
        f"      per-row send:   "
        f"min={min(_row_send_seconds)*1000:.1f} ms "
        f"mean={mean(_row_send_seconds)*1000:.1f} ms median={median(_row_send_seconds)*1000:.1f} ms "
        f"max={max(_row_send_seconds)*1000:.1f} ms"
    )
    print(
        f"      per-row wait (ack):   "
        f"min={min(_row_wait_seconds)*1000:.1f} ms "
        f"mean={mean(_row_wait_seconds)*1000:.1f} ms median={median(_row_wait_seconds)*1000:.1f} ms "
        f"max={max(_row_wait_seconds)*1000:.1f} ms"
    )
    print(
        f"      per-row send+wait:   "
        f"min={min(_row_ack_seconds)*1000:.1f} ms "
        f"mean={mean(_row_ack_seconds)*1000:.1f} ms median={median(_row_ack_seconds)*1000:.1f} ms "
        f"max={max(_row_ack_seconds)*1000:.1f} ms"
    )
if _batch_run_seconds and _batch_n:
    _ms_row_4b = (mean(_batch_run_seconds) / _batch_n) * 1000
    print(
        f"  4b ({_batch_n} batched, {len(_batch_run_seconds)} runs):"
        f"  wall  min={min(_batch_run_seconds)*1000:.1f} ms"
        f"  mean={mean(_batch_run_seconds)*1000:.1f} ms"
        f"  median={median(_batch_run_seconds)*1000:.1f} ms"
        f"  max={max(_batch_run_seconds)*1000:.1f} ms"
    )
    print(
        f"      send:       "
        f"min={min(_batch_send_seconds)*1000:.1f} ms  "
        f"mean={mean(_batch_send_seconds)*1000:.1f} ms  "
        f"median={median(_batch_send_seconds)*1000:.1f} ms  "
        f"max={max(_batch_send_seconds)*1000:.1f} ms"
    )
    print(
        f"      wait (ack): "
        f"min={min(_batch_wait_seconds)*1000:.1f} ms  "
        f"mean={mean(_batch_wait_seconds)*1000:.1f} ms  "
        f"median={median(_batch_wait_seconds)*1000:.1f} ms  "
        f"max={max(_batch_wait_seconds)*1000:.1f} ms"
    )
if _singles and _batch_run_seconds and _batch_n:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    _ms_row_4b = (mean(_batch_run_seconds) / _batch_n) * 1000
    _ratio = _ms_row_4a / _ms_row_4b
    print("  batch vs single latency comparison")
    print(f"    4a ({_singles} singles) {_ms_row_4a:.1f} wall amortized ms/row")
    print(f"    4b (1 batch of {_batch_n} rows) {_ms_row_4b:.3f} wall amortized ms/row  (mean over {len(_batch_run_seconds)} runs)")
    print(f"    4a/4b = ~{_ratio:.1f}x lower ms/row")

Before ingest: count=921630 min(device_name)='sensor-0' max(device_name)='sensor-999'
Ingested 1000 rows into main.robert_lee.airquality_http2_async  [mode=http2_async]
After ingest:  expected_count=931540 visible_count=931540
  visibility: 9866.5 ms  (from first row send → COUNT(*) reached target)
  visibility: 5743.1 ms  (from end of ingest → COUNT(*) reached target)
After visibility: min(device_name)='sensor-0' max(device_name)='sensor-999'
oauth:        277.6 ms
connect:      28.8 ms
disconnect:   0.2 ms
  Ingest+wait wall (4a+4b only; excludes stream.close): 3990.6 ms
  4a (10 singles): wall 2109.7 ms  (~211.0 ms/row amortized)
      per-row send:   min=200.5 ms mean=210.9 ms median=201.5 ms max=297.2 ms
      per-row wait (ack):   min=0.0 ms mean=0.0 ms median=0.0 ms max=0.0 ms
      per-row send+wait:   min=200.5 ms mean=210.9 ms median=201.5 ms max=297.2 ms
  4b (990 batched, 10 runs):  wall  min=69.7 ms  mean=188.1 ms  median=200.9 ms  max=206.4 ms
      send:       min=69.7 m

## Step 4.e: Append benchmark result

Appends one JSON record to `benchmark_results.jsonl` (same directory as this notebook). Each record captures the run timestamp, mode, all 4a/4b timing distributions, and lakehouse visibility. After appending, the cell loads the full file with `pandas.read_json(..., lines=True)` and displays it as a table.

In [0]:
import datetime
import json
import os
from pathlib import Path
from statistics import mean, median

# File written alongside this notebook. Override by setting ZEROBUS_RESULTS_DIR.
_RESULTS_FILE = Path(os.environ.get("ZEROBUS_RESULTS_DIR", Path(__file__).parent if "__file__" in dir() else Path.cwd())).expanduser() / "benchmark_results.jsonl"

_MODE_LABEL = {
    "grpc_sync":   "gRPC sync",
    "grpc_async":  "gRPC async",
    "http_sync":   "HTTP/1.1 sync",
    "http_async":  "HTTP/1.1 async",
    "http2_sync":  "HTTP/2 sync",
    "http2_async": "HTTP/2 async",
}


def _ms(seconds: float) -> float:
    return round(seconds * 1000, 3)


def _stats_ms(values: list) -> dict | None:
    if not values:
        return None
    return {
        "min":    _ms(min(values)),
        "mean":   _ms(mean(values)),
        "median": _ms(median(values)),
        "max":    _ms(max(values)),
    }


_result = {
    "run_at":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "mode":             MODE,
    "mode_label":       _MODE_LABEL.get(MODE, MODE),
    "table":            TABLE_NAME,
    "zerobus_endpoint": ZEROBUS_INGEST_URL,
    "n":                _n,
    "concurrency":      _concurrency,
    "oauth_ms":      _ms(_oauth_s)      if _oauth_s      is not None else None,
    "ping_ms":       _ms(_ping_s),
    "connect_ms":    _ms(_connect_s)    if _connect_s    is not None else None,
    "disconnect_ms": _ms(_disconnect_s) if _disconnect_s is not None else None,
    "ingest_wall_ms":  _ms(_ingest_4a4b_s),
    "visibility_from_first_send_ms": _ms(_visibility_from_first_send_s),
    "visibility_from_end_ms":        _ms(_visibility_s),
    "4a": {
        "rows":    1,          # each call sends 1 row
        "runs":    _singles,   # number of single-row calls (matches _n=1000 → _singles=10)
        "wall_ms": _ms(_singles_wall_s),
        "send_ms":  _stats_ms(_row_send_seconds),
        "wait_ms":  _stats_ms(_row_wait_seconds),
        "send_wait_ms": _stats_ms(_row_ack_seconds),
    } if _singles else None,
    "4b": {
        "rows":    _batch_n,
        "runs":    len(_batch_run_seconds),
        "wall_ms": _ms(sum(_batch_run_seconds)),  # total wall for all runs (scalar, mirrors 4a)
        "send_wait_ms": _stats_ms(_batch_run_seconds),  # per-run send+wait distribution
        "send_ms":      _stats_ms(_batch_send_seconds),
        "wait_ms":      _stats_ms(_batch_wait_seconds),
    } if _batch_run_seconds else None,
}

_RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(_RESULTS_FILE, "a") as _fh:
    _fh.write(json.dumps(_result) + "\n")

print(f"Appended to {_RESULTS_FILE}  ({_RESULTS_FILE.stat().st_size} bytes total)")
print(json.dumps(_result, indent=2))


Appended to benchmark_results.jsonl  (1027 bytes total)
{
  "run_at": "2026-04-22T19:17:01.807205+00:00",
  "mode": "http2_async",
  "mode_label": "HTTP/2 async",
  "table": "main.robert_lee.airquality_http2_async",
  "zerobus_endpoint": "https://1444828305810485.zerobus.us-west-2.cloud.databricks.com",
  "n": 1000,
  "concurrency": 1,
  "oauth_ms": 277.606,
  "ping_ms": 74.827,
  "connect_ms": 28.79,
  "disconnect_ms": 0.214,
  "ingest_wall_ms": 3990.588,
  "visibility_from_first_send_ms": 9866.486,
  "visibility_from_end_ms": 5743.131,
  "4a": {
    "rows": 1,
    "runs": 10,
    "wall_ms": 2109.726,
    "send_ms": {
      "min": 200.494,
      "mean": 210.912,
      "median": 201.455,
      "max": 297.157
    },
    "wait_ms": {
      "min": 0.0,
      "mean": 0.0,
      "median": 0.0,
      "max": 0.0
    },
    "send_wait_ms": {
      "min": 200.494,
      "mean": 210.912,
      "median": 201.455,
      "max": 297.157
    }
  },
  "4b": {
    "rows": 990,
    "runs": 10,
    "wall

In [0]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

_COMMON = [
    "run_at", "mode", "mode_label", "table", "zerobus_endpoint", "n", "concurrency",
    "oauth_ms", "ping_ms", "connect_ms", "disconnect_ms",
    "ingest_wall_ms", "visibility_from_first_send_ms", "visibility_from_end_ms",
]


def _expand_stats(rec: dict, key: str, src) -> None:
    """Flatten a {'min','mean','median','max'} dict into key_min, key_mean, … columns."""
    if isinstance(src, dict):
        for stat in ("min", "mean", "median", "max"):
            rec[f"{key}_{stat}"] = src.get(stat)


def _flatten(raw: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in raw.iterrows():
        base = {c: r.get(c) for c in _COMMON}
        for label in ("4a", "4b"):
            d = r.get(label)
            if not isinstance(d, dict):
                continue
            rec = {**base, "scenario": label}
            rec["rows"] = d.get("rows")
            rec["runs"] = d.get("runs")
            # wall_ms: total wall for all runs in this scenario (scalar).
            # 4a = total for all single-row calls; 4b = total for all batch runs.
            wall = d.get("wall_ms")
            if wall is not None:
                rec["wall_ms"] = wall
            _expand_stats(rec, "send_wait_ms", d.get("send_wait_ms"))  # send+wait per call (both 4a and 4b)
            _expand_stats(rec, "send_ms",  d.get("send_ms"))
            _expand_stats(rec, "wait_ms",  d.get("wait_ms"))
            rows.append(rec)
    return pd.DataFrame(rows)


_df = _flatten(pd.read_json(_RESULTS_FILE, lines=True))

# When a mode has multiple runs, aggregate numeric columns by median.
# run_at is dropped (not meaningful across runs); run_count shows how many were merged.
_GROUP = ["mode", "scenario"]
_run_counts = _df.groupby(_GROUP).size().reset_index(name="run_count")
if _run_counts["run_count"].max() > 1:
    _num  = _df.select_dtypes(include="number").columns.tolist()
    _strs = [c for c in _df.columns if c not in _num and c not in _GROUP and c != "run_at"]
    _df = (
        _df.groupby(_GROUP, sort=False)
        .agg({**{c: "median" for c in _num}, **{c: "first" for c in _strs}})
        .reset_index()
        .merge(_run_counts, on=_GROUP)
    )
    _df = _df.drop(columns=["run_at"], errors="ignore")
    _FRONT = ["scenario", "rows", "runs", "run_count", "mode_label"]
else:
    _FRONT = ["scenario", "rows", "runs", "mode_label", "run_at"]

_REST = [c for c in _df.columns if c not in _FRONT]
_df = _df[_FRONT + _REST]
display(_df)

scenario,rows,runs,mode_label,run_at,mode,table,zerobus_endpoint,n,concurrency,oauth_ms,ping_ms,connect_ms,disconnect_ms,ingest_wall_ms,visibility_from_first_send_ms,visibility_from_end_ms,wall_ms,send_wait_ms_min,send_wait_ms_mean,send_wait_ms_median,send_wait_ms_max,send_ms_min,send_ms_mean,send_ms_median,send_ms_max,wait_ms_min,wait_ms_mean,wait_ms_median,wait_ms_max
4a,1,10,HTTP/2 async,2026-04-22T19:17:01.807Z,http2_async,main.robert_lee.airquality_http2_async,https://1444828305810485.zerobus.us-west-2.cloud.databricks.com,1000,1,277.606,74.827,28.79,0.214,3990.588,9866.486,5743.131,2109.726,200.494,210.912,201.455,297.157,200.494,210.912,201.455,297.157,0.0,0.0,0.0,0.0
4b,990,10,HTTP/2 async,2026-04-22T19:17:01.807Z,http2_async,main.robert_lee.airquality_http2_async,https://1444828305810485.zerobus.us-west-2.cloud.databricks.com,1000,1,277.606,74.827,28.79,0.214,3990.588,9866.486,5743.131,1880.862,69.691,188.086,200.902,206.375,69.691,188.086,200.902,206.375,0.0,0.0,0.0,0.0


### Acknowledgment callback

**gRPC modes only.** `setup_zerobus` registers a `DemoAckCallback` on `StreamConfigurationOptions(ack_callback=…)` (subclass `AckCallback`; implement `on_ack` / `on_error`). During 4a and 4b you should see `[ack callback] offset … acknowledged` lines as the service acks offsets. Accumulated events are stored in `_zb_client["_ack_events"]`. HTTP modes have no ack callback — the HTTP POST itself is the acknowledgment.

### Protocol Buffers

**gRPC modes only.** For type-safe ingestion, use **Protocol Buffers** with `RecordType.PROTO` (default) and provide a `descriptor_proto` in table properties. See the **[Python SDK repository](https://github.com/databricks/zerobus-sdk-py)** for `generate_proto`, batch ingestion, and full configuration options.

In [0]:
# Sample rows (marketing: each row is often ack'd in ~250ms; full table typically visible within a few seconds)
display(spark.sql(f"SELECT * FROM {TABLE_NAME} ORDER BY device_name LIMIT 20"))

device_name,temp,humidity
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50
sensor-0,20,50


# Step 5: Zerobus system tables

Zerobus **ingest** and **stream** telemetry in `system.lakeflow` (Beta; see [Zerobus system tables](https://docs.databricks.com/aws/en/admin/system-tables/zerobus-ingest)). Rows can lag; tables must be enabled for your workspace.


#### 5.a. `zerobus_ingest`

All columns from `system.lakeflow.zerobus_ingest` for this table (`SELECT *`, filtered by `table_name` and `workspace_id`, `ORDER BY commit_time DESC`, `LIMIT 20`).


In [0]:
# Step 5a: commit batches (zerobus_ingest)
# there will be some delays

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_ingest
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY commit_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus Ingest system table.")
    else:
        raise

commit_version,stream_id,workspace_id,account_id,table_id,table_name,commit_time,committed_bytes,committed_records,tags,errors
140,46d30200-fe42-49c2-85aa-193cce8bb338,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-22T17:48:13.225Z,387450,6930,List(),List()
139,46d30200-fe42-49c2-85aa-193cce8bb338,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-22T17:48:08.276Z,166590,2980,List(),List()
138,19d8ad08-61f8-4677-bb8f-0a660b61de6b,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-22T17:39:58.258Z,554040,9910,List(),List()
137,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:48.444Z,554040,9910,List(),List()
136,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:43.460Z,554040,9910,List(),List()
135,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:38.424Z,110700,1980,List(),List()
134,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:33.385Z,443340,7930,List(),List()
133,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:28.450Z,332100,5940,List(),List()
132,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:23.398Z,221940,3970,List(),List()
131,b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,2026-04-21T22:31:19.054Z,553500,9900,List(),List()


#### 5.b. `zerobus_stream`

All columns from `system.lakeflow.zerobus_stream` for this table (same `table_name` / `workspace_id` filters, `LIMIT 20`).


In [0]:
# Step 5b: stream events (zerobus_stream)

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_stream
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY event_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_stream is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus stream system table.")
    else:
        raise

stream_id,event_time,workspace_id,account_id,producer_id,opened_time,closed_time,table_id,table_name,protocol,data_format,errors
46d30200-fe42-49c2-85aa-193cce8bb338,2026-04-22T17:48:05.246Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-22T17:48:05.246Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
19d8ad08-61f8-4677-bb8f-0a660b61de6b,2026-04-22T17:39:52.945Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-22T17:39:52.945Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
b6cf6ccb-0a55-4dce-b1bf-179e76e3efdc,2026-04-21T22:31:02.265Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:31:02.265Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
3e64282f-f00c-4f72-8998-84f72866b241,2026-04-21T22:28:02.212Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:28:02.212Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
d8d4f1a9-cff5-4197-932b-6d06fb81703c,2026-04-21T22:25:02.763Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:25:02.763Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
62f8a69a-17eb-45a9-8792-30761463a4b5,2026-04-21T22:22:07.348Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:22:07.348Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
6a60a2df-5a3f-4c62-89d4-967bc4a5e666,2026-04-21T22:19:22.530Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:19:22.530Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
2e6f5122-b0ec-496e-8274-0dfd714da089,2026-04-21T22:16:12.216Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:16:12.216Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
d4ac8ded-149b-4cf6-af34-7735912552f8,2026-04-21T22:13:21.821Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:13:21.821Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
af821d33-d0a6-4e99-ac62-864b4e2fe00c,2026-04-21T22:10:32.255Z,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,null,2026-04-21T22:10:32.255Z,null,91a3181b-6852-4dfc-b8c7-0d30fca4bd5b,main.robert_lee.airquality_http2_async,HTTP,JSON,List()
